# ⚕️ Clinical RAG Assistant — Colab Walkthrough

Retrieval-Augmented Generation over clinical reference text: chunking → embeddings → Chroma retrieval → grounded generation with page citations.

This notebook is deliberately thin — every pipeline step calls the repo's own `scripts/` and `src/` code, so it always runs exactly the same code as the repository and can't drift out of sync.

**One honest shortcut:** the full project indexes a 4,100-page manual that isn't in the repo. Here we index a tiny 2-page sample PDF generated below, so the whole walkthrough runs in a few minutes. The code path is identical — only the corpus is smaller. Step 5 shows how to run the full pipeline.

## 1 · Setup

Clone the repo and install dependencies. Safe to re-run.

In [ ]:
import os

REPO = "clinical-rag-assistant"
if not os.path.isdir(REPO):
    !git clone -q https://github.com/rtwumasiankrah/clinical-rag-assistant.git
if os.path.basename(os.getcwd()) != REPO:
    os.chdir(REPO)
!pip install -q -r requirements.txt

## 2 · Build a mini index

We generate a 2-page sample clinical PDF, then index it with the repo's real `build_index` entrypoint — the same command the full pipeline uses. (First run downloads the `thenlper/gte-large` embedding model, ~1 GB.)

In [ ]:
import fitz  # PyMuPDF

SEPSIS = ("Sepsis is life-threatening organ dysfunction caused by a dysregulated host "
    "response to infection. Early recognition saves lives. For adults with suspected sepsis, "
    "obtain blood cultures before starting antibiotics, but do not delay antibiotics more than "
    "45 minutes to do so. Administer broad-spectrum antimicrobials within one hour of recognition. "
    "Begin rapid crystalloid fluid resuscitation for hypotension or serum lactate of 4 mmol/L or "
    "greater. Reassess frequently and narrow antibiotics once the pathogen is identified.")

APPENDICITIS = ("Acute appendicitis is inflammation of the vermiform appendix, usually from luminal "
    "obstruction by a fecalith or lymphoid hyperplasia. Classic presentation begins with vague "
    "periumbilical pain that migrates to the right lower quadrant within 24 hours, accompanied by "
    "anorexia, nausea, and low-grade fever. Diagnosis is clinical, supported by an elevated white "
    "blood cell count and imaging such as ultrasound or CT. Definitive treatment is appendectomy; "
    "uncomplicated cases may be managed with antibiotics alone in selected patients.")

doc = fitz.open()
for title, body in [("Sepsis", SEPSIS), ("Acute Appendicitis", APPENDICITIS)]:
    page = doc.new_page()
    page.insert_text((72, 72), title, fontsize=18)
    page.insert_textbox(fitz.Rect(72, 110, 540, 720), body, fontsize=11)
doc.save("data/sample_manual.pdf")
print("sample PDF written: data/sample_manual.pdf")

In [ ]:
!python scripts/build_index.py --pdf data/sample_manual.pdf --persist-dir mini_db --chunk-size 200 --chunk-overlap 20

## 3 · Retrieve grounded context

Ask a question against the mini index using the repo's own `retrieve_context` — the same function the Q&A path uses. Note the page numbers attached to every hit: those become the citations.

In [ ]:
from pathlib import Path
from src.config import RAGConfig
from src.qa import retrieve_context

config = RAGConfig()
config.persist_dir = Path("mini_db")

question = "When should antibiotics be given for sepsis?"
hits = retrieve_context(question, config)
for h in hits:
    print(f"--- page {h['page']} (distance {h['distance']:.3f}) ---")
    print(h["text"][:350] + "...\n")

## 4 · See the exact prompt the LLM receives

The repo's strict system prompt forces the model to answer *only* from the retrieved context — "I don't know" beats a hallucination. This is the real `build_qa_prompt` output for the hits above.

In [ ]:
from src.prompts import build_qa_prompt

context = ". ".join(h["text"] for h in hits)
print(build_qa_prompt(context=context, question=question))

## 5 · Going full-scale

The walkthrough above ran the real pipeline on a sample. For the full project:

```bash
# 1. add your copy of the 4,100-page manual (see data/README.md)
# 2. build the vector index (once — this is the slow step)
python scripts/build_index.py --pdf data/medical_diagnosis_manual.pdf

# 3. ask a question (Mistral-7B weights download automatically on first run)
python scripts/ask.py "What is the protocol for managing sepsis in a critical care unit?"

# or launch the interactive demo
streamlit run app/streamlit_app.py
```

> ⚠️ Educational prototype — not medical advice.